# Train / Test 


The split is chronological: the test set is the last 20% of the rows and we don't touch it until the end. The test set starts on 2025-05-16 (144 rows, train has 578).

The target doesn't look ahead anymore (it ends at the close of day `t`), so we don't need to remove rows of train. We keep `gap = 1` in `TimeSeriesSplit`.

In [19]:
import pandas as pd
import numpy as np 
from sklearn.model_selection import TimeSeriesSplit

In [20]:
data = pd.read_feather("../data/final_data_ny.feather")

In [21]:
n_test = int(len(data) * 0.2)
TEST_START = data.index[-n_test]
train = data.loc[data.index < TEST_START]
test = data.loc[TEST_START:]

In [22]:
gap = 1

In [23]:
print("train:", train.index.min(), "->", train.index.max(), len(train))
print("test: ", test.index.min(), "->", test.index.max(), len(test))
print("TEST_START:", TEST_START.date())

train: 2023-01-26 00:00:00 -> 2025-05-15 00:00:00 578
test:  2025-05-16 00:00:00 -> 2025-12-11 00:00:00 144
TEST_START: 2025-05-16


In [24]:
test.to_feather("../data/test_holdout_ny.feather")

Class balance: in train the NY session goes up 55.5% of the days (56.9% in test). So a model that always says "up" already gets that accuracy, that's the baseline to beat.

In [25]:
target_cols = [("ret_ny", ""), ("y_ny", "")]
feature_cols = [c for c in data.columns if c not in target_cols]

In [26]:
tscv = TimeSeriesSplit(n_splits=5, gap=gap)
X_train = train.drop(columns=target_cols)
y_train = train[[("y_ny", "")]]

## Checks

In [27]:
print("final shape:", data.shape)
print("train:", train.shape, train.index.min().date(), "->", train.index.max().date())
print("test: ", test.shape, test.index.min().date(), "->", test.index.max().date())

final shape: (722, 128)
train: (578, 128) 2023-01-26 -> 2025-05-15
test:  (144, 128) 2025-05-16 -> 2025-12-11


In [28]:
print([c for c in feature_cols if c[0] != "stock_ret"])
print((data.columns.get_level_values(0) == "stock_ret").sum(), "stock_ret columns")

[('Dayofweek', ''), ('ndx_ret_1d', ''), ('ndx_ret_5d', ''), ('ndx_ret_20d', ''), ('ndx_ret_ny_prev', ''), ('ndx_vol_20d', ''), ('vix', ''), ('vxn', ''), ('yield_10y', ''), ('yield_3m', ''), ('dollar_ret', ''), ('oil_ret', ''), ('gold_ret', ''), ('btc_ret', ''), ('btc_vol_7d', ''), ('pct_up', ''), ('asia_ret', ''), ('europe_ret', ''), ('ovn_ret', ''), ('asia_high_dist', ''), ('asia_low_dist', ''), ('europe_high_dist', ''), ('europe_low_dist', ''), ('ovn_high_dist', ''), ('ovn_low_dist', ''), ('ovn_range_pos', ''), ('prev_ny_high_dist', ''), ('prev_ny_low_dist', ''), ('ovn_volume_rel', ''), ('asia_volume_share', ''), ('europe_volume_share', ''), ('last_hour_volume_rel', ''), ('ovn_vol', ''), ('ovn_range', '')]
92 stock_ret columns


In [29]:
print(train[("y_ny", "")].value_counts(normalize=True))
print(test[("y_ny", "")].value_counts(normalize=True))

(y_ny, )
1.0    0.555363
0.0    0.444637
Name: proportion, dtype: float64
(y_ny, )
1.0    0.569444
0.0    0.430556
Name: proportion, dtype: float64


Leakage sanity check: correlation of each feature with the target return on train (NQ features first, then the daily features). A value above 0.3 (in absolute value) would be suspicious.

In [30]:
# the NQ overnight features (built in 1_Project_Features)
nq_names = ["asia_ret", "europe_ret", "ovn_ret", "asia_high_dist", "asia_low_dist",
            "europe_high_dist", "europe_low_dist", "ovn_high_dist", "ovn_low_dist", "ovn_range_pos",
            "prev_ny_high_dist", "prev_ny_low_dist", "ovn_volume_rel", "asia_volume_share",
            "europe_volume_share", "last_hour_volume_rel", "ovn_vol", "ovn_range"]
nq_cols = [(n, "") for n in nq_names]
assert all(c in data.columns for c in nq_cols)

In [31]:
corr = train[nq_cols].corrwith(train[("ret_ny", "")])
corr.index = [c[0] for c in corr.index]
print(corr.sort_values())
print("max |corr|:", corr.abs().max())

asia_ret               -0.116855
europe_low_dist        -0.081548
asia_low_dist          -0.063654
ovn_low_dist           -0.063242
ovn_ret                -0.058614
asia_volume_share      -0.054364
ovn_range_pos          -0.028815
europe_ret              0.025273
ovn_volume_rel          0.042284
asia_high_dist          0.043686
europe_volume_share     0.054364
last_hour_volume_rel    0.060476
prev_ny_low_dist        0.081445
ovn_high_dist           0.097493
ovn_range               0.122001
europe_high_dist        0.126657
prev_ny_high_dist       0.157110
ovn_vol                 0.249443
dtype: float64
max |corr|: 0.24944336881790719


In [32]:
daily_cols = [c for c in feature_cols if c[0] != "stock_ret" and c not in nq_cols]
corr_daily = train[daily_cols].corrwith(train[("ret_ny", "")])
corr_daily.index = [c[0] for c in corr_daily.index]
print(corr_daily.sort_values())
print("max |corr|:", corr_daily.abs().max())

ndx_ret_ny_prev   -0.162624
btc_ret           -0.069112
ndx_ret_1d        -0.063381
ndx_ret_5d        -0.060737
Dayofweek         -0.057932
yield_10y         -0.052061
ndx_ret_20d       -0.043620
pct_up            -0.036114
oil_ret           -0.026860
yield_3m          -0.025598
dollar_ret        -0.009711
gold_ret          -0.009263
btc_vol_7d         0.014104
ndx_vol_20d        0.069231
vxn                0.095714
vix                0.142906
dtype: float64
max |corr|: 0.16262354262429152


**Known limitation:** the table has one row per day and the NQ file only covers Dec 2022 - Dec 2025, so we only have 722 rows (578 for train and 144 for test). It is small. Possible options for later: a longer NQ source (since 2008, not on Kaggle), or a panel with one row per stock per day.